## Naive Bayes 
- Probability thresholds 
- How Optuna works 
If you want to predict why, you can look at the probability of (y): what's the probability of predicting class 1 or class 2 
- no boosting happening 
- straight statistical prediction
- look at each variable separately
- each feature is independent

## GaussianNB
- all the values for class 1 are placed together 
- do the same thing for the other class 
- average all the values, find that distance 
- there is overlap: likelihood for cancer vs. not likelihood for cancer
- oversimplified math still is a good predictor 

## Other Naive Bayes Models 
- GaussianNB
    - For numerical (continuous) features
- BernoulliNB 
    - For binary features (0/1)
- Multinominal NB
    - For count data
    - Frequency based features

## Changing the threshold 
- Make sure to choose the right metric when changing the threshold 
- precision, recall, accuracy = biggest noteable difference when changing the threshold 

## How Optuna works 
- tries parameters across your defined ranges
- uses bayesian approach to identify which regions work well
- deeper search in better regions
- occasionally tries random values to avoid getting stuck

In [30]:
# importing libraries
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, balanced_accuracy_score



In [31]:
irrigation = pd.read_csv('/Users/diyapatel/Documents/GitHub/Advanced-ML/Homework-4/In class activities/Data/train.csv')

In [32]:
# inspect the data
print(irrigation.head())
print()

print(irrigation["Irrigation_Need"].value_counts())
print()

print("missing values:", irrigation.isna().sum().sum())

   id Soil_Type  Soil_pH  Soil_Moisture  Organic_Carbon  \
0   0     Loamy     4.92          32.58            1.01   
1   1      Clay     7.08          56.61            0.44   
2   2      Clay     5.69          27.71            0.81   
3   3     Sandy     5.65          13.32            1.33   
4   4      Clay     7.96          59.14            0.38   

   Electrical_Conductivity  Temperature_C  Humidity  Rainfall_mm  \
0                     3.05          15.01     50.61       725.99   
1                     2.00          22.92     67.86       985.66   
2                     2.83          26.97     92.22      2201.70   
3                     0.87          13.32     61.57      1357.33   
4                     0.96          20.22     91.11      1538.20   

   Sunlight_Hours  ...  Crop_Type Crop_Growth_Stage  Season Irrigation_Type  \
0            5.90  ...  Sugarcane            Sowing    Zaid            Drip   
1            6.98  ...      Wheat        Vegetative  Kharif         Rainfed   

In [33]:
# features and target
X_irrigation = irrigation.drop("Irrigation_Need", axis=1)
y_irrigation = irrigation["Irrigation_Need"]

# encode target
le = LabelEncoder()
y_irrigation = le.fit_transform(y_irrigation)

# convert categorical feature columns to numeric
X_irrigation = pd.get_dummies(X_irrigation, drop_first=False)

# split AFTER encoding
X_train_irrigation, X_test_irrigation, y_train_irrigation, y_test_irrigation = train_test_split(
    X_irrigation,
    y_irrigation,
    test_size=0.20,
    stratify=y_irrigation,
    random_state=42
)

In [34]:
# defining functions
# helper function for cross validation comparison
def compare_models_cv(models,X_train,y_train,cv):
    cv_results=[]
    for name,model in models.items():
        scores=cross_validate(model,X_train,y_train,cv=cv,scoring=["accuracy","precision_weighted","recall_weighted","f1_weighted","balanced_accuracy"])
        cv_results.append({
            "model":name,
            "cv_accuracy":scores["test_accuracy"].mean(),
            "cv_precision":scores["test_precision_weighted"].mean(),
            "cv_recall":scores["test_recall_weighted"].mean(),
            "cv_f1":scores["test_f1_weighted"].mean(),
            "cv_balanced_accuracy":scores["test_balanced_accuracy"].mean()
        })
    return pd.DataFrame(cv_results).sort_values("cv_balanced_accuracy",ascending=False)

In [35]:
# helper function for threshold tuning
def evaluate_thresholds(y_true,probs,thresholds=None):
    if thresholds is None:
        thresholds=np.linspace(0.1,0.9,17) # change 17 to higher number to test more thresholds
    rows=[]
    for t in thresholds:
        preds=(probs[:,1]>=t).astype(int)
        rows.append({
            "threshold":t,
            "precision":precision_score(y_true,preds,zero_division=0, average='weighted'),
            "recall":recall_score(y_true,preds,zero_division=0, average='weighted'),
            "f1":f1_score(y_true,preds,zero_division=0, average='weighted'),
            "balanced_accuracy":balanced_accuracy_score(y_true,preds)
        })
    return pd.DataFrame(rows)

In [36]:
# helper function for held out test evaluation
def evaluate_models_test(models,X_train,X_test,y_train,y_test):
    trained_models={}
    test_results=[]
    for name,model in models.items():
        model.fit(X_train,y_train)
        trained_models[name]=model
        preds=model.predict(X_test)
        test_results.append({
            "model":name,
            "test_accuracy":accuracy_score(y_test,preds),
            "test_precision":precision_score(y_test,preds,zero_division=0, average='weighted'),
            "test_recall":recall_score(y_test,preds,zero_division=0, average='weighted'),
            "test_f1":f1_score(y_test,preds,zero_division=0, average='weighted'),
            "test_balanced_accuracy":balanced_accuracy_score(y_test,preds)
        })
    return trained_models,pd.DataFrame(test_results).sort_values("test_balanced_accuracy",ascending=False)

In [37]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models_irrigation = {
    "Naive Bayes": GaussianNB()
}

In [38]:
# Baseline model evaluation on test data
# fit model on full training data and evaluate on held-out test set 

trained_models_irrigation,test_results_irrigation=evaluate_models_test(
    models_irrigation,X_train_irrigation,X_test_irrigation,y_train_irrigation,y_test_irrigation
)
test_results_irrigation

,model,test_accuracy,test_precision,test_recall,test_f1,test_balanced_accuracy
0,Naive Bayes,0.746119,0.743579,0.746119,0.734423,0.564208


In [40]:
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, f1_score, recall_score, balanced_accuracy_score

# -----------------------------
# 1. Get model probabilities
# -----------------------------
model = trained_models_irrigation["Naive Bayes"]
probs_irrigation_nb = model.predict_proba(X_test_irrigation)

# show label mapping
label_mapping = dict(zip(le.classes_, le.transform(le.classes_)))
print("Label mapping:", label_mapping)
print()

# -----------------------------
# 2. Baseline predictions
# default rule = choose class with highest probability
# -----------------------------
baseline_preds = np.argmax(probs_irrigation_nb, axis=1)

print("BASELINE RESULTS")
print("Weighted F1:", round(f1_score(y_test_irrigation, baseline_preds, average="weighted", zero_division=0), 4))
print("Balanced Accuracy:", round(balanced_accuracy_score(y_test_irrigation, baseline_preds), 4))
print()
print(classification_report(y_test_irrigation, baseline_preds, target_names=le.classes_, zero_division=0))

Label mapping: {'High': np.int64(0), 'Low': np.int64(1), 'Medium': np.int64(2)}

BASELINE RESULTS
Weighted F1: 0.7344
Balanced Accuracy: 0.5642

              precision    recall  f1-score   support

        High       0.84      0.23      0.36      4202
         Low       0.77      0.88      0.82     73983
      Medium       0.70      0.58      0.63     47815

    accuracy                           0.75    126000
   macro avg       0.77      0.56      0.61    126000
weighted avg       0.74      0.75      0.73    126000



In [42]:
# -----------------------------
# 4. Tune threshold for the chosen class
# metric = recall for the focus class
# -----------------------------
thresholds = np.linspace(0.1, 0.9, 17)

rows = []

for t in thresholds:
    adjusted_preds = []
    
    for i in range(len(probs_irrigation_nb)):
        # if probability for focus class is high enough, force that class
        if probs_irrigation_nb[i, focus_class_index] >= t:
            adjusted_preds.append(focus_class_index)
        else:
            # otherwise choose the best class among the remaining classes
            adjusted_preds.append(np.argmax(probs_irrigation_nb[i]))
    
    adjusted_preds = np.array(adjusted_preds)
    
    rows.append({
        "threshold": t,
        "focus_class_f1": f1_score(
            (y_test_irrigation == focus_class_index).astype(int),
            (adjusted_preds == focus_class_index).astype(int),
            zero_division=0
        ),
        "focus_class_recall": recall_score(
            (y_test_irrigation == focus_class_index).astype(int),
            (adjusted_preds == focus_class_index).astype(int),
            zero_division=0
        ),
        "overall_balanced_accuracy": balanced_accuracy_score(y_test_irrigation, adjusted_preds)
    })

threshold_results_irrigation = pd.DataFrame(rows)

print("THRESHOLD SEARCH RESULTS")
print(threshold_results_irrigation.sort_values("focus_class_recall", ascending=False).head())

THRESHOLD SEARCH RESULTS
   threshold  focus_class_f1  focus_class_recall  overall_balanced_accuracy
0       0.10        0.392601            0.866254                   0.710712
1       0.15        0.453768            0.688482                   0.681766
2       0.20        0.475056            0.526892                   0.645185
3       0.25        0.474968            0.435745                   0.622791
4       0.30        0.464164            0.372204                   0.606268


In [50]:
best = threshold_results_irrigation.sort_values("focus_class_f1", ascending=False).iloc[0]

best_threshold = best["threshold"]

print()
print("Best threshold:", round(best_threshold, 4))
print("Best focus-class recall:", round(best["focus_class_recall"], 4))
print("Focus-class F1 at best threshold:", round(best["focus_class_f1"], 4))
print("Overall balanced accuracy at best threshold:", round(best["overall_balanced_accuracy"], 4))


Best threshold: 0.2
Best focus-class recall: 0.5269
Focus-class F1 at best threshold: 0.4751
Overall balanced accuracy at best threshold: 0.6452


In [51]:
threshold_preds = []

for i in range(len(probs_irrigation_nb)):
    if probs_irrigation_nb[i, focus_class_index] >= best_threshold:
        threshold_preds.append(focus_class_index)
    else:
        # exclude High when below threshold
        probs_copy = probs_irrigation_nb[i].copy()
        probs_copy[focus_class_index] = -1  # remove High
        threshold_preds.append(np.argmax(probs_copy))

threshold_preds = np.array(threshold_preds)

In [52]:
# -----------------------------
# 6. Compare baseline vs threshold-adjusted predictions
# -----------------------------
print("BASELINE vs THRESHOLD-ADJUSTED")
print()

print("Baseline weighted F1:",
      round(f1_score(y_test_irrigation, baseline_preds, average="weighted", zero_division=0), 4))
print("Threshold weighted F1:",
      round(f1_score(y_test_irrigation, threshold_preds, average="weighted", zero_division=0), 4))
print()

print("Baseline balanced accuracy:",
      round(balanced_accuracy_score(y_test_irrigation, baseline_preds), 4))
print("Threshold balanced accuracy:",
      round(balanced_accuracy_score(y_test_irrigation, threshold_preds), 4))
print()

baseline_focus_recall = recall_score(
    (y_test_irrigation == focus_class_index).astype(int),
    (baseline_preds == focus_class_index).astype(int),
    zero_division=0
)

threshold_focus_recall = recall_score(
    (y_test_irrigation == focus_class_index).astype(int),
    (threshold_preds == focus_class_index).astype(int),
    zero_division=0
)

baseline_focus_f1 = f1_score(
    (y_test_irrigation == focus_class_index).astype(int),
    (baseline_preds == focus_class_index).astype(int),
    zero_division=0
)

threshold_focus_f1 = f1_score(
    (y_test_irrigation == focus_class_index).astype(int),
    (threshold_preds == focus_class_index).astype(int),
    zero_division=0
)

print(f"Baseline recall for {focus_class_name}:", round(baseline_focus_recall, 4))
print(f"Threshold recall for {focus_class_name}:", round(threshold_focus_recall, 4))
print()

print(f"Baseline F1 for {focus_class_name}:", round(baseline_focus_f1, 4))
print(f"Threshold F1 for {focus_class_name}:", round(threshold_focus_f1, 4))
print()

BASELINE vs THRESHOLD-ADJUSTED

Baseline weighted F1: 0.7344
Threshold weighted F1: 0.7262

Baseline balanced accuracy: 0.5642
Threshold balanced accuracy: 0.6452

Baseline recall for High: 0.2301
Threshold recall for High: 0.5269

Baseline F1 for High: 0.3609
Threshold F1 for High: 0.4751



In [54]:
print("THRESHOLD-ADJUSTED CLASSIFICATION REPORT")
print(classification_report(y_test_irrigation, threshold_preds, target_names=le.classes_, zero_division=0))

THRESHOLD-ADJUSTED CLASSIFICATION REPORT
              precision    recall  f1-score   support

        High       0.43      0.53      0.48      4202
         Low       0.77      0.88      0.82     73983
      Medium       0.71      0.53      0.60     47815

    accuracy                           0.74    126000
   macro avg       0.64      0.65      0.63    126000
weighted avg       0.73      0.74      0.73    126000



In [55]:
probs_test = final_model.predict_proba(X_test_kaggle)

threshold_preds_kaggle = []

for i in range(len(probs_test)):
    if probs_test[i, focus_class_index] >= best_threshold:
        threshold_preds_kaggle.append(focus_class_index)
    else:
        probs_copy = probs_test[i].copy()
        probs_copy[focus_class_index] = -1
        threshold_preds_kaggle.append(np.argmax(probs_copy))

threshold_preds_kaggle = np.array(threshold_preds_kaggle)

# convert back to labels
preds_kaggle_labels = le.inverse_transform(threshold_preds_kaggle)

submission = pd.DataFrame({
    "id": test_ids,
    "Irrigation_Need": preds_kaggle_labels
})

submission.to_csv("naive_bayes_submission.csv", index=False)

Using a markdown cell, discuss which class you selected, what threshold you chose, how the metric changed, what tradeoff you observed, and how Naive Bayes compares to your existing model(s).  You should use a classification report to assist with this discussion.

I selected the 'High' irrigation need class as my focus. In farming, failing to identify a 'High' need for irrigation is much more high stakes or important that can lead to crop failure. Since it's the minority class, my baseline model struggled to identify it. After testing a range of threshold, I chose a threshold of 0.20 because it provided the best balance, maximizing the focus-class F1 score.

Based on the classification reports, shifting from the default threshold to 0.20 gave major changes. The model became much better at actually finding 'High' irrigation need. The Recall for the 'High' class increased from .23 to .53 and its F1 score increased from .36 to .48. The overall Balanced Accuracy also increased from .54 to .645. Although the model captured more 'High' need classes there was a bigger drop in precision. Precision for 'High' dropped down to .43, which means the model is producing more false positives where its predicting 'High' when the actual need is 'Medium' or 'low'. 

Compared to the other ensemble models I submitted into Kaggle, like the XGBoost and CatBoost models, this Naive Bayes model is much weaker with multi-class classification problems. This helped me understand that boosting models can actually capture complex interaction features with Temperature, Humidity or Soil Moisture. This can be because Naive Bayes assumes all these features are independent as discussed in class, which is unrealistic. Although the performance was not as good with an overall Kaggle score of 0.654, Naive Bayes did run much faster compareed to the other boosting models.